# Vígil.ia — YOLO11n (nano): ainda menor que o 11s

Testa a hipótese "menor generaliza melhor no vídeo" um degrau abaixo: o **yolo11n**
(~2,6 M params, ~6,5 GFLOPs — 3,6× menor que o 11s). Mesma receita/parâmetros do 11s,
medido no vídeo (out-of-domain).

**Toggle `FROM_SCRATCH`:**
- `False` (padrão, recomendado): parte dos pesos **COCO** (`yolo11n.pt`) → fine-tune v3.
  É o "do zero" do nosso pipeline (igual s/l/x).
- `True`: init **aleatório** (`yolo11n.yaml`, sem COCO). Literalmente do zero absoluto —
  ⚠️ tende a ficar **bem pior** em dataset pequeno; precisa de muito mais épocas.

> ⚠️ Enquanto só houver vídeo de intacto, o ranking mede só falso-alarme (pode
> premiar viés-intacto). Reconfirme com os vídeos de defeito.
> Resumível; não sobrescreve nada (salva `soja_yolo11n[_scratch]_<receita>.pt`).

## 0. Setup

In [ ]:
!pip -q install "ultralytics==8.4.80"

import torch, ultralytics
ultralytics.checks()
assert torch.cuda.is_available(), 'Sem GPU! (Ambiente de execução -> Alterar tipo -> GPU/A100)'
print('GPU:', torch.cuda.get_device_name(0))

## 1. Caminhos + modo de treino

In [ ]:
import os
from google.colab import drive
drive.mount('/content/drive')

REAL_SRCS = [
    '/content/drive/MyDrive/Soja total/Soja total/Lotes',
    '/content/drive/MyDrive/Soja pra completar',
]
VAL_ROOT = '/content/drive/MyDrive/Vídeos para treino/Treino'
assert os.path.isdir(VAL_ROOT), f'VAL_ROOT não existe: {VAL_ROOT}'

# ===== escolha o modo =====
FROM_SCRATCH = False   # False: COCO->v3 (recomendado) | True: init aleatório (sem COCO)

if FROM_SCRATCH:
    BASE_WEIGHTS = 'yolo11n.yaml'   # arquitetura crua, pesos aleatórios
    EPOCHS = 300                    # scratch precisa de MUITO mais épocas
    SUFFIX = 'scratch_'
    print('MODO: init ALEATÓRIO (sem COCO) — ⚠️ deve ficar pior; ~horas por receita')
else:
    BASE_WEIGHTS = 'yolo11n.pt'     # pré-treinado COCO (recomendado)
    EPOCHS = 60
    SUFFIX = ''
    print('MODO: COCO -> v3 (recomendado)')
print('base:', BASE_WEIGHTS, '| épocas:', EPOCHS)

## 2. Dataset v3 (real + sintético) — reconstrói se a sessão for nova

In [ ]:
import glob, hashlib, unicodedata, cv2, yaml
import numpy as np

NAMES = ['broken', 'immature', 'intact', 'skin-damaged', 'spotted']
ALIASES = {0: ['broken', 'quebrad'], 1: ['immature', 'imatur', 'nao maduro'],
           2: ['intact'], 3: ['skin', 'casca', 'ardid', 'danific'], 4: ['spotted', 'manchad']}
IGNORE = ['part of the original']
IMG_EXT = ('.jpg', '.jpeg', '.png', '.bmp', '.webp')
RNG = np.random.default_rng(42)

def norm(s):
    return unicodedata.normalize('NFKD', s).encode('ascii', 'ignore').decode().lower()

def class_of(folder):
    n = norm(folder)
    if any(norm(k) in n for k in IGNORE):
        return None
    for idx in range(5):
        if any(norm(k) in n for k in ALIASES[idx]):
            return idx
    return None

def collect_real(srcs, val_frac=0.15):
    items = []
    for src in srcs:
        for root, _, files in os.walk(src):
            cls = None
            for part in reversed(root.split(os.sep)):
                c = class_of(part)
                if c is not None:
                    cls = c; break
            if cls is None:
                continue
            for fn in files:
                if fn.lower().endswith(IMG_EXT):
                    p = os.path.join(root, fn)
                    h = int(hashlib.md5(p.encode()).hexdigest(), 16)
                    items.append((p, cls, 'val' if (h % 100) < val_frac * 100 else 'train'))
    from collections import Counter
    print('coletado:', dict(Counter(sp for _, _, sp in items)))
    return items

def sat_box(img):
    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
    s = cv2.GaussianBlur(hsv[:, :, 1], (5, 5), 0)
    _, th = cv2.threshold(s, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    th = cv2.morphologyEx(th, cv2.MORPH_OPEN, np.ones((5, 5), np.uint8))
    cnts, _ = cv2.findContours(th, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not cnts:
        return None
    c = max(cnts, key=cv2.contourArea)
    area = cv2.contourArea(c)
    h, w = img.shape[:2]
    if area < 0.01 * h * w or area > 0.90 * h * w:
        return None
    x, y, bw, bh = cv2.boundingRect(c)
    pad = int(0.04 * min(bw, bh)) + 2
    x1, y1 = max(0, x - pad), max(0, y - pad)
    x2, y2 = min(w, x + bw + pad), min(h, y + bh + pad)
    return (((x1 + x2) / 2) / w, ((y1 + y2) / 2) / h, (x2 - x1) / w, (y2 - y1) / h)

def otsu_box(img):
    h, w = img.shape[:2]
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    blur = cv2.GaussianBlur(gray, (5, 5), 0)
    _, th = cv2.threshold(blur, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    cnts, _ = cv2.findContours(th, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not cnts:
        return None
    c = max(cnts, key=cv2.contourArea)
    area = cv2.contourArea(c)
    if area < 0.005 * h * w or area > 0.995 * h * w:
        return None
    x, y, bw, bh = cv2.boundingRect(c)
    pad = int(0.04 * min(bw, bh)) + 2
    x1, y1 = max(0, x - pad), max(0, y - pad)
    x2, y2 = min(w, x + bw + pad), min(h, y + bh + pad)
    return (((x1 + x2) / 2) / w, ((y1 + y2) / 2) / h, (x2 - x1) / w, (y2 - y1) / h)

def letterbox640(img, size=640):
    h, w = img.shape[:2]
    s = size / max(h, w)
    img = cv2.resize(img, (max(1, round(w * s)), max(1, round(h * s))))
    h, w = img.shape[:2]
    top, left = (size - h) // 2, (size - w) // 2
    img = cv2.copyMakeBorder(img, top, size - h - top, left, size - w - left,
                             cv2.BORDER_CONSTANT, value=(0, 0, 0))
    return img, s, left, top

def motion_blur(img, rng=RNG):
    k = int(rng.choice([7, 9, 11, 13, 15]))
    kernel = np.zeros((k, k), np.float32)
    kernel[k // 2, :] = 1.0
    M = cv2.getRotationMatrix2D((k / 2 - 0.5, k / 2 - 0.5), float(rng.uniform(0, 180)), 1)
    kernel = cv2.warpAffine(kernel, M, (k, k))
    kernel /= max(kernel.sum(), 1e-6)
    return cv2.filter2D(img, -1, kernel)

def extract_cutout(img):
    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
    s = cv2.GaussianBlur(hsv[:, :, 1], (5, 5), 0)
    _, th = cv2.threshold(s, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    th = cv2.morphologyEx(th, cv2.MORPH_OPEN, np.ones((5, 5), np.uint8))
    cnts, _ = cv2.findContours(th, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not cnts:
        return None
    c = max(cnts, key=cv2.contourArea)
    area = cv2.contourArea(c)
    h, w = img.shape[:2]
    if area < 0.01 * h * w or area > 0.90 * h * w:
        return None
    mask = np.zeros((h, w), np.uint8)
    cv2.drawContours(mask, [c], -1, 255, -1)
    x, y, bw, bh = cv2.boundingRect(c)
    return img[y:y + bh, x:x + bw], mask[y:y + bh, x:x + bw]

def make_scene(cutouts, rng=RNG, size=640):
    bg = int(rng.integers(20, 130))
    canvas = np.clip(np.full((size, size, 3), bg, np.int16)
                     + rng.normal(0, 6, (size, size, 3)), 0, 255).astype(np.uint8)
    occ = np.zeros((size, size), np.uint8)
    boxes = []
    for _ in range(int(rng.integers(6, 26))):
        cls, crop, mask = cutouts[int(rng.integers(len(cutouts)))]
        s = int(rng.integers(60, 150)) / max(crop.shape[:2])
        crop2 = cv2.resize(crop, None, fx=s, fy=s)
        mask2 = cv2.resize(mask, None, fx=s, fy=s, interpolation=cv2.INTER_NEAREST)
        h2, w2 = crop2.shape[:2]
        diag = int(np.ceil(np.hypot(h2, w2))) + 2
        M = cv2.getRotationMatrix2D((w2 / 2, h2 / 2), float(rng.uniform(0, 360)), 1)
        M[0, 2] += (diag - w2) / 2
        M[1, 2] += (diag - h2) / 2
        crop3 = cv2.warpAffine(crop2, M, (diag, diag))
        mask3 = cv2.warpAffine(mask2, M, (diag, diag), flags=cv2.INTER_NEAREST)
        ys, xs = np.where(mask3 > 0)
        if not len(xs):
            continue
        crop3 = crop3[ys.min():ys.max() + 1, xs.min():xs.max() + 1]
        mask3 = mask3[ys.min():ys.max() + 1, xs.min():xs.max() + 1]
        gh, gw = mask3.shape
        if gh >= size - 2 or gw >= size - 2:
            continue
        placed = False
        for _try in range(20):
            px = int(rng.integers(0, size - gw))
            py = int(rng.integers(0, size - gh))
            inter = (occ[py:py + gh, px:px + gw] > 0) & (mask3 > 0)
            if inter.sum() <= 0.15 * (mask3 > 0).sum():
                placed = True
                break
        if not placed:
            continue
        alpha = (cv2.GaussianBlur(mask3, (5, 5), 0).astype(np.float32) / 255)[..., None]
        reg = canvas[py:py + gh, px:px + gw]
        canvas[py:py + gh, px:px + gw] = (alpha * crop3 + (1 - alpha) * reg).astype(np.uint8)
        occ[py:py + gh, px:px + gw][mask3 > 0] = 255
        boxes.append((cls, (px + gw / 2) / size, (py + gh / 2) / size, gw / size, gh / size))
    return canvas, boxes

def balance_train(items):
    from collections import defaultdict
    train = [it for it in items if it[2] == 'train']
    rest = [it for it in items if it[2] != 'train']
    by = defaultdict(list)
    for it in train:
        by[it[1]].append(it)
    mx = max(len(v) for v in by.values())
    out = []
    for c, v in by.items():
        out += v + [v[int(i)] for i in RNG.integers(0, len(v), mx - len(v))]
    print('balanceado (train):', {NAMES[c]: sum(1 for it in out if it[1] == c) for c in sorted(by)})
    return out + rest

def build_v3(items, out_dir, n_synth=600, blur_frac=0.4):
    assert items, 'Nenhuma imagem coletada! Confira REAL_SRCS.'
    for sp in ('train', 'val', 'test'):
        os.makedirs(f'{out_dir}/images/{sp}', exist_ok=True)
        os.makedirs(f'{out_dir}/labels/{sp}', exist_ok=True)
    items = balance_train(items)
    cutouts = []
    kept = skipped = 0
    for i, (path, cls, sp) in enumerate(items):
        if i % 200 == 0:
            print(f'  fotos {i}/{len(items)}…', flush=True)
        img = cv2.imread(path)
        if img is None:
            skipped += 1; continue
        h0, w0 = img.shape[:2]
        box = sat_box(img) or otsu_box(img)
        if box is None:
            skipped += 1; continue
        if sp == 'train':
            cut = extract_cutout(img)
            if cut is not None:
                cutouts.append((cls, cut[0], cut[1]))
        lb, s, left, top = letterbox640(img)
        cx, cy, ww, hh = box
        cx = (cx * w0 * s + left) / 640.0
        cy = (cy * h0 * s + top) / 640.0
        ww = (ww * w0 * s) / 640.0
        hh = (hh * h0 * s) / 640.0
        line = f'{cls} {cx:.6f} {cy:.6f} {ww:.6f} {hh:.6f}'
        stem = f'{sp}_{i:06d}'
        cv2.imwrite(f'{out_dir}/images/{sp}/{stem}.jpg', lb, [cv2.IMWRITE_JPEG_QUALITY, 95])
        open(f'{out_dir}/labels/{sp}/{stem}.txt', 'w').write(line)
        kept += 1
        if sp == 'train':
            cv2.imwrite(f'{out_dir}/images/train/{stem}b.jpg', motion_blur(lb),
                        [cv2.IMWRITE_JPEG_QUALITY, 95])
            open(f'{out_dir}/labels/train/{stem}b.txt', 'w').write(line)
            kept += 1
    print(f'fotos reais: kept={kept} skipped={skipped} | recortes: {len(cutouts)}')
    assert cutouts, 'Nenhum recorte extraído!'
    synth = 0
    for j in range(n_synth):
        if j % 100 == 0:
            print(f'  cenas {j}/{n_synth}…', flush=True)
        canvas, boxes = make_scene(cutouts)
        if not boxes:
            continue
        if RNG.random() < blur_frac:
            canvas = motion_blur(canvas)
        stem = f'synth_{j:05d}'
        cv2.imwrite(f'{out_dir}/images/train/{stem}.jpg', canvas, [cv2.IMWRITE_JPEG_QUALITY, 95])
        open(f'{out_dir}/labels/train/{stem}.txt', 'w').write(
            '\n'.join(f'{c} {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}' for c, cx, cy, w, h in boxes))
        synth += 1
    print(f'cenas sintéticas: {synth}')
    yaml.safe_dump({'path': out_dir, 'train': 'images/train', 'val': 'images/val',
                    'test': 'images/test', 'names': {i: n for i, n in enumerate(NAMES)}},
                   open(f'{out_dir}/data.yaml', 'w'), sort_keys=False, allow_unicode=True)
    return f'{out_dir}/data.yaml'

SPLIT_MAP = {'train': 'train', 'valid': 'val', 'val': 'val', 'test': 'test'}

def collect_base(base_dir):
    """Acha train/valid/test em qualquer profundidade dentro do dataset 12,5k."""
    items = []
    for root, dirs, _ in os.walk(base_dir):
        for d in list(dirs):
            sp = SPLIT_MAP.get(d.lower())
            if sp is None:
                continue
            split_dir = os.path.join(root, d)
            for folder in sorted(os.listdir(split_dir)):
                cls = class_of(folder)
                if cls is None:
                    continue
                for p_ in glob.glob(os.path.join(split_dir, folder, '*')):
                    if p_.lower().endswith(IMG_EXT):
                        items.append((p_, cls, sp))
            dirs.remove(d)
    from collections import Counter
    print('coletado (base 12,5k):', dict(Counter(sp for _, _, sp in items)))
    return items

def build_base(items, out_dir):
    """Dataset base de detecção: pseudo-rótulo Otsu (1 grão/img, fundo preto),
    sem balanceamento/blur/sintético — idêntico ao estágio base do RT-DETR."""
    assert items, 'Nenhuma imagem do 12,5k coletada!'
    for sp in ('train', 'val', 'test'):
        os.makedirs(f'{out_dir}/images/{sp}', exist_ok=True)
        os.makedirs(f'{out_dir}/labels/{sp}', exist_ok=True)
    kept = skipped = 0
    for i, (path, cls, sp) in enumerate(items):
        if i % 1000 == 0:
            print(f'  base {i}/{len(items)}…', flush=True)
        img = cv2.imread(path)
        if img is None:
            skipped += 1; continue
        h0, w0 = img.shape[:2]
        box = otsu_box(img)
        if box is None:
            skipped += 1; continue
        lb, s, left, top = letterbox640(img)
        cx, cy, ww, hh = box
        cx = (cx * w0 * s + left) / 640.0
        cy = (cy * h0 * s + top) / 640.0
        ww = (ww * w0 * s) / 640.0
        hh = (hh * h0 * s) / 640.0
        stem = f'{sp}_{i:06d}'
        cv2.imwrite(f'{out_dir}/images/{sp}/{stem}.jpg', lb, [cv2.IMWRITE_JPEG_QUALITY, 95])
        open(f'{out_dir}/labels/{sp}/{stem}.txt', 'w').write(
            f'{cls} {cx:.6f} {cy:.6f} {ww:.6f} {hh:.6f}')
        kept += 1
    print(f'base: kept={kept} skipped={skipped}')
    yaml.safe_dump({'path': out_dir, 'train': 'images/train', 'val': 'images/val',
                    'test': 'images/test', 'names': {i: n for i, n in enumerate(NAMES)}},
                   open(f'{out_dir}/data.yaml', 'w'), sort_keys=False, allow_unicode=True)
    return f'{out_dir}/data.yaml'

# só o dataset v3 (fine-tune) — este notebook não usa a base 12,5k
V3_YAML = '/content/soja_det_v3/data.yaml'
if not os.path.exists(V3_YAML):
    V3_YAML = build_v3(collect_real(REAL_SRCS), '/content/soja_det_v3')
print('dataset v3:', V3_YAML)

## 3. Treino do 11n — mesma receita/parâmetros do 11s

Todas as receitas partem do mesmo `BASE_WEIGHTS`, mesmo batch/seed/épocas — só muda
o que está em `RECIPES` (idêntico ao sweep do 11s, pra comparar direto).

In [ ]:
from ultralytics import YOLO
import shutil

COMMON = dict(
    imgsz=640, device=0, seed=42, optimizer='AdamW',
    cache=False, workers=8,
    mosaic=1.0, degrees=15, translate=0.1, scale=0.5, fliplr=0.5, flipud=0.5,
    project='runs_11n', exist_ok=True,
)
BATCH = 32   # mesmo do sweep do 11s (comparação justa)

RECIPES = {
    'baseline':  dict(box=7.5,  hsv_v=0.5),
    'box10':     dict(box=10.0, hsv_v=0.5),
    'aug':       dict(box=7.5,  hsv_h=0.05, hsv_s=0.8, hsv_v=0.6),
    'box10_aug': dict(box=10.0, hsv_h=0.05, hsv_s=0.8, hsv_v=0.6),
}

variant_pt = {}
for name, extra in RECIPES.items():
    dst = f'/content/drive/MyDrive/soja_yolo11n_{SUFFIX}{name}.pt'
    if os.path.exists(dst):
        print(f'{name}: já existe ({dst}) — pulando'); variant_pt[name] = dst; continue
    print(f'\n{"="*56}\n11n / {name}: {extra}\n{"="*56}')
    m = YOLO(BASE_WEIGHTS)
    m.train(name=f'11n_{SUFFIX}{name}', data=V3_YAML, batch=BATCH,
            epochs=EPOCHS, lr0=0.001, patience=max(20, EPOCHS // 4),
            close_mosaic=8, **{**COMMON, **extra})
    shutil.copy(str(m.trainer.best), dst)
    variant_pt[name] = dst
    print(f'ok: {os.path.basename(dst)}')

print('\nvariantes:', variant_pt)

## 4. Avaliação no VÍDEO (out-of-domain)
Por detecção, quadro a quadro. Só pastas com vídeo.

In [ ]:
from collections import Counter
from ultralytics import YOLO

variant_pt = {name: f'/content/drive/MyDrive/soja_yolo11n_{SUFFIX}{name}.pt'
              for name in ('baseline', 'box10', 'aug', 'box10_aug')
              if os.path.exists(f'/content/drive/MyDrive/soja_yolo11n_{SUFFIX}{name}.pt')}
assert variant_pt, 'Nenhum soja_yolo11n_*.pt no Drive — rode o §3 antes.'

VIDEO_EXT = ('.mp4', '.mov', '.avi', '.mkv')
by_class = {}
for entry in sorted(os.listdir(VAL_ROOT)):
    sub = os.path.join(VAL_ROOT, entry)
    if not os.path.isdir(sub):
        continue
    c = class_of(entry)
    if c is None:
        continue
    vids = [os.path.join(sub, f) for f in sorted(os.listdir(sub))
            if f.lower().endswith(VIDEO_EXT)]
    if vids:
        by_class[NAMES[c]] = vids
print('classes com vídeo:', {k: len(v) for k, v in by_class.items()})
assert by_class, 'Nenhum vídeo por classe em VAL_ROOT.'

def video_eval(pt, vid_stride=5):
    model = YOLO(pt)
    tot, cert = Counter(), Counter()
    for true_cls, vids in by_class.items():
        for vid in vids:
            for r in model.predict(source=vid, imgsz=640, conf=0.35, iou=0.5,
                                   agnostic_nms=True, vid_stride=vid_stride,
                                   stream=True, verbose=False):
                for c in r.boxes.cls.int().tolist():
                    tot[true_cls] += 1
                    if NAMES[c] == true_cls:
                        cert[true_cls] += 1
    return tot, cert

video_acc = {}
for name, pt in variant_pt.items():
    tot, cert = video_eval(pt)
    n, ok = sum(tot.values()), sum(cert.values())
    video_acc[name] = 100 * ok / max(n, 1)
    porcl = {c: f'{100*cert[c]/tot[c]:.0f}%' for c in tot}
    print(f'{name:10s}: {ok}/{n} = {video_acc[name]:.1f}%  | por classe: {porcl}')

print('\n===== RANKING 11n no vídeo =====')
for name in sorted(video_acc, key=lambda k: -video_acc[k]):
    print(f'  {name:10s} {video_acc[name]:.1f}%')
print('\n(compare com o sweep do 11s — 11n tem ~2,6M vs 9,4M params)')

## 5. (contexto) mAP no val v3 — in-domain

In [ ]:
from ultralytics import YOLO

print('mAP no val v3 (in-domain — só contexto; o juiz é o vídeo):\n')
for name, pt in variant_pt.items():
    r = YOLO(pt).val(data=V3_YAML, split='val', imgsz=640, device=0, verbose=False)
    print(f'  {name:10s} mAP50={r.box.map50:.3f}  mAP50-95={r.box.map:.3f}')

print('\n⚠️ Ranking provisório enquanto só houver vídeo de intacto (pode ser viés-')
print('   intacto). Reconfirme com broken/spotted/skin-damaged/immature.')